# Notebook 2 — HelaBERT Embedding & Semantic Analysis

**Datasets:**
- A: `SinLlama-Dataset.txt`          (10.7 M lines, 3.0 GB)
- B: `CPT-Dataset.txt` (8.7 M lines, 2.4 GB)

**Model:** HelaBERT Large (`BertModel`, 768-dim CLS embeddings, 110 M params)

**Strategy:**
| Computation | Data used |
|---|---|
| Centroid (mean embedding) | Full corpus — accumulated in O(1) memory |
| MMD | 10 K reservoir subsample per dataset |
| UMAP scatter / density | 20 K reservoir subsample per dataset |
| Pairwise similarity | 5 K random pairs from reservoir |

The reservoir is filled during the single embedding pass via **Algorithm R** (uniform random sample, no prior knowledge of N required).


## 1. Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import umap
import json, random, warnings
from pathlib import Path
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from transformers import AutoModel
import sentencepiece as spm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.dpi"] = 130

# ── paths ──────────────────────────────────────────────────────────────────────
BASE       = Path("/ml/SinLlama_CPT/data_analysis")
MODEL_PATH = BASE / "HelaBERT_Large/HelaBERT_Large"
SP_MODEL   = BASE / "HelaBERT_Large/tokenizer/unigram_32000_0.9995.model"
FILE_A     = BASE / "SinLlama-Dataset.txt"
FILE_B     = BASE / "CPT-Dataset.txt"
OUT_DIR    = BASE / "outputs"
OUT_DIR.mkdir(exist_ok=True)

# ── embedding-pass config ──────────────────────────────────────────────────────
BATCH_SIZE     = 512    # safe for RTX 4060 8 GB in fp32
MAX_LEN        = 128    # avg line is ~22 tokens; 128 covers >99 % of lines
RESERVOIR_SIZE = 50_000 # kept per dataset; used for MMD, UMAP, pairwise sims
HIDDEN_DIM     = 768
CLS_ID, SEP_ID, PAD_ID = 2, 3, 0

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Load HelaBERT

In [ ]:
model = AutoModel.from_pretrained(str(MODEL_PATH))
model = model.to(device).eval()
sp    = spm.SentencePieceProcessor(); sp.Load(str(SP_MODEL))

n_params = sum(p.numel() for p in model.parameters())
print(f"HelaBERT loaded: {n_params:,} parameters")
print(f"Vocab size     : {sp.GetPieceSize()}")


## 3. Tokenisation helper

In [ ]:
def tokenize_batch(lines, max_len=MAX_LEN):
    """SentencePiece → padded input_ids + attention_mask tensors."""
    all_ids, all_masks = [], []
    for line in lines:
        ids  = [CLS_ID] + sp.EncodeAsIds(line.strip())[:max_len - 2] + [SEP_ID]
        mask = [1] * len(ids)
        all_ids.append(ids)
        all_masks.append(mask)
    max_L = max(len(x) for x in all_ids)
    for i in range(len(all_ids)):
        pad = max_L - len(all_ids[i])
        all_ids[i]   += [PAD_ID] * pad
        all_masks[i] += [0]      * pad
    return (torch.tensor(all_ids,   dtype=torch.long),
            torch.tensor(all_masks, dtype=torch.long))


## 4. Full-corpus embedding pass  ⏱ *long running*

Three separate cells — run them in order. Each saves its own checkpoint immediately on completion, so if a cell is interrupted you can re-run it without losing the other dataset. **If the checkpoints already exist in `outputs/`, each cell loads them instead of recomputing** (the full pass takes ~5–6 h per dataset).

| Cell | What it does | Checkpoint saved |
|---|---|---|
| **4a** | Defines helpers + embeds Dataset A (SinLlama) | `centroid_A.npy`, `embeddings_A_sample.npy`, `embedding_stats.json` (`n_a`) |
| **4b** | Embeds Dataset B (CPT-Dataset) | `centroid_B.npy`, `embeddings_B_sample.npy`, `embedding_stats_B.json` (`n_total`/`n_b`) |
| **4c** | Loads both checkpoints, frees GPU memory | — |

> **Note:** A's line count is in `embedding_stats.json` (key `n_a`); B's is in a separate `embedding_stats_B.json` (key `n_total`/`n_b`). The split is legacy — Dataset A (SinLlama, the larger corpus) was embedded in a standalone cloud-GPU run and its outputs were copied into the A slot. Cells 4b/4c detect either layout.

In [ ]:
### 4a — Helpers + Embed Dataset A
import time

CKPT = {
    "centroid_a": OUT_DIR / "centroid_A.npy",
    "centroid_b": OUT_DIR / "centroid_B.npy",
    "sample_a"  : OUT_DIR / "embeddings_A_sample.npy",
    "sample_b"  : OUT_DIR / "embeddings_B_sample.npy",
    "stats"     : OUT_DIR / "embedding_stats.json",
}

KNOWN_LINES = {
    str(FILE_A): 10_730_154,
    str(FILE_B): 8_696_658,
}

def embed_dataset(filepath, label):
    """Stream one file, return (centroid float32, n_total int, reservoir_sample float32)."""
    centroid_sum = np.zeros(HIDDEN_DIM, dtype=np.float64)
    n_total      = 0
    reservoir    = np.zeros((RESERVOIR_SIZE, HIDDEN_DIM), dtype=np.float32)
    buf          = []
    t_start      = time.time()

    def flush(lines):
        nonlocal centroid_sum, n_total
        ids, masks = tokenize_batch(lines)
        with torch.no_grad():
            out  = model(input_ids=ids.to(device), attention_mask=masks.to(device))
            embs = out.last_hidden_state[:, 0, :].float().cpu().numpy()
        centroid_sum += embs.sum(axis=0)
        for k, emb in enumerate(embs):         # Algorithm R reservoir sampling
            g = n_total + k
            if g < RESERVOIR_SIZE:
                reservoir[g] = emb
            else:
                j = random.randint(0, g)
                if j < RESERVOIR_SIZE:
                    reservoir[j] = emb
        n_total += len(embs)

    with open(filepath, encoding="utf-8", errors="ignore") as f:
        pbar = tqdm(f, desc=f"Embedding {label}",
                    total=KNOWN_LINES.get(str(filepath)),
                    unit="line", mininterval=5.0,
                    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")
        for line in pbar:
            line = line.strip()
            if not line:
                continue
            buf.append(line)
            if len(buf) == BATCH_SIZE:
                flush(buf); buf.clear()
        if buf:
            flush(buf)

    elapsed   = time.time() - t_start
    hrs, rem  = divmod(int(elapsed), 3600)
    mins, sec = divmod(rem, 60)
    print(f"  ✓ {label}: {n_total:,} lines in {hrs:02d}h {mins:02d}m {sec:02d}s "
          f"| {n_total/elapsed:,.0f} lines/s")

    centroid = (centroid_sum / n_total).astype(np.float32)
    sample   = reservoir[:min(n_total, RESERVOIR_SIZE)].copy()
    return centroid, int(n_total), sample

# ── run or load ────────────────────────────────────────────────────────────────
if CKPT["centroid_a"].exists() and CKPT["sample_a"].exists():
    centroid_a = np.load(CKPT["centroid_a"])
    embs_a     = np.load(CKPT["sample_a"])
    n_a        = json.loads(CKPT["stats"].read_text()).get("n_a", -1) if CKPT["stats"].exists() else -1
    print(f"A: loaded from checkpoint — centroid {centroid_a.shape}, sample {embs_a.shape}")
else:
    centroid_a, n_a, embs_a = embed_dataset(FILE_A, "A (SinLlama)")
    np.save(CKPT["centroid_a"], centroid_a)
    np.save(CKPT["sample_a"],   embs_a)
    existing = json.loads(CKPT["stats"].read_text()) if CKPT["stats"].exists() else {}
    existing["n_a"] = n_a
    CKPT["stats"].write_text(json.dumps(existing))
    print(f"A: checkpoint saved — centroid {centroid_a.shape}, sample {embs_a.shape}")

In [ ]:
### 4b — Embed Dataset B
# Dataset B was embedded by a standalone run that wrote its line count to a
# separate file (embedding_stats_B.json, key "n_total") instead of merging it
# into embedding_stats.json. Detect either layout here.
STATS_B = OUT_DIR / "embedding_stats_B.json"

if CKPT["centroid_b"].exists() and CKPT["sample_b"].exists():
    centroid_b = np.load(CKPT["centroid_b"])
    embs_b     = np.load(CKPT["sample_b"])
    if STATS_B.exists():
        _d  = json.loads(STATS_B.read_text())
        n_b = int(_d.get("n_b", _d.get("n_total", -1)))
    elif CKPT["stats"].exists():
        n_b = int(json.loads(CKPT["stats"].read_text()).get("n_b", -1))
    else:
        n_b = -1
    print(f"B: loaded from checkpoint — centroid {centroid_b.shape}, sample {embs_b.shape}, N_B={n_b:,}")
else:
    centroid_b, n_b, embs_b = embed_dataset(FILE_B, "B (CPT-Dataset)")
    np.save(CKPT["centroid_b"], centroid_b)
    np.save(CKPT["sample_b"],   embs_b)
    # write both keys so either loader (4b/4c) finds it
    STATS_B.write_text(json.dumps({"n_b": n_b, "n_total": n_b, "label": "B (CPT-Dataset)"}))
    existing = json.loads(CKPT["stats"].read_text()) if CKPT["stats"].exists() else {}
    existing["n_b"] = n_b
    CKPT["stats"].write_text(json.dumps(existing))
    print(f"B: checkpoint saved — centroid {centroid_b.shape}, sample {embs_b.shape}")

In [ ]:
### 4c — Verify both + free GPU memory
STATS_B = OUT_DIR / "embedding_stats_B.json"
stats_a = json.loads(CKPT["stats"].read_text()) if CKPT["stats"].exists() else {}
stats_b = json.loads(STATS_B.read_text())       if STATS_B.exists()       else {}

n_a = int(stats_a.get("n_a", len(embs_a)))
n_b = int(stats_b.get("n_b", stats_b.get("n_total", stats_a.get("n_b", len(embs_b)))))

torch.cuda.empty_cache()

print(f"centroid_a : {centroid_a.shape}   centroid_b : {centroid_b.shape}")
print(f"embs_a     : {embs_a.shape}    embs_b     : {embs_b.shape}")
print(f"N_A = {n_a:,}   N_B = {n_b:,}")
print("\nGPU memory freed. Ready for analysis cells 5–11.")

## 5. Centroid distance

In [ ]:
ca = centroid_a / np.linalg.norm(centroid_a)
cb = centroid_b / np.linalg.norm(centroid_b)

cosine_sim  = float(np.dot(ca, cb))
cosine_dist = 1.0 - cosine_sim
l2_dist     = float(np.linalg.norm(centroid_a - centroid_b))
angle_deg   = float(np.degrees(np.arccos(np.clip(cosine_sim, -1, 1))))

print(f"Cosine similarity  : {cosine_sim:.6f}")
print(f"Cosine distance    : {cosine_dist:.6f}")
print(f"L2 distance        : {l2_dist:.6f}")
print(f"Angle between      : {angle_deg:.3f}°")


## 6. PCA centroid arrow plot
Projects 50 K combined embeddings to 2D via PCA. Overlays centroid points with an arrow showing direction of semantic shift A → B.

In [ ]:
PCA_SAMPLE = 25_000   # per dataset for scatter (GPU-free, fast)

rng  = np.random.default_rng(42)
ia   = rng.choice(len(embs_a), min(PCA_SAMPLE, len(embs_a)), replace=False)
ib   = rng.choice(len(embs_b), min(PCA_SAMPLE, len(embs_b)), replace=False)

combined = np.vstack([embs_a[ia], embs_b[ib],
                      centroid_a[None], centroid_b[None]])

pca      = PCA(n_components=2, random_state=42)
proj     = pca.fit_transform(combined)
var_exp  = pca.explained_variance_ratio_ * 100

pa, pb        = proj[:len(ia)], proj[len(ia):len(ia)+len(ib)]
pca_cent_a    = proj[-2]
pca_cent_b    = proj[-1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pa[:, 0], pa[:, 1], s=2, alpha=0.15, color="#4C72B0", label="A (SinLlama)",  rasterized=True)
ax.scatter(pb[:, 0], pb[:, 1], s=2, alpha=0.15, color="#DD8452", label="B (CPT-Dataset)", rasterized=True)
ax.scatter(*pca_cent_a, s=180, color="#1a3a6b", zorder=5, marker="*", label="Centroid A")
ax.scatter(*pca_cent_b, s=180, color="#8b3a00", zorder=5, marker="*", label="Centroid B")
ax.annotate("", xy=pca_cent_b, xytext=pca_cent_a,
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.8))

ax.set_xlabel(f"PC 1 ({var_exp[0]:.1f} % var)")
ax.set_ylabel(f"PC 2 ({var_exp[1]:.1f} % var)")
ax.set_title("PCA Projection — Centroid Shift A → B")
ax.legend(markerscale=4, framealpha=0.9)
plt.tight_layout()
plt.savefig(OUT_DIR / "04_centroid_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/04_centroid_pca.png")


## 7. Distance-from-centroid histogram
Shows how tightly each dataset clusters around its own centroid. Narrow = compact semantic space; wide = dispersed.

In [ ]:
ea_norm = normalize(embs_a)
eb_norm = normalize(embs_b)
ca_norm = centroid_a / np.linalg.norm(centroid_a)
cb_norm = centroid_b / np.linalg.norm(centroid_b)

dist_a = 1.0 - (ea_norm @ ca_norm)   # cosine distance from own centroid
dist_b = 1.0 - (eb_norm @ cb_norm)

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, max(dist_a.max(), dist_b.max()) + 0.01, 80)
ax.hist(dist_a, bins=bins, alpha=0.6, color="#4C72B0", label=f"A  mean={dist_a.mean():.4f}", density=True)
ax.hist(dist_b, bins=bins, alpha=0.6, color="#DD8452", label=f"B  mean={dist_b.mean():.4f}", density=True)
ax.set_xlabel("Cosine distance from own centroid")
ax.set_ylabel("Density")
ax.set_title("Embedding Compactness — Distance from Centroid")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "05_centroid_distance_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/05_centroid_distance_hist.png")


## 8. Maximum Mean Discrepancy (MMD)
RBF kernel with the **median heuristic** for bandwidth σ. Computed on a 10 K subsample from each reservoir (full kernel matrix = 10 K × 10 K, ~400 MB).

In [ ]:
MMD_N = 10_000

rng  = np.random.default_rng(0)
xa   = embs_a[rng.choice(len(embs_a), MMD_N, replace=False)].astype(np.float32)
xb   = embs_b[rng.choice(len(embs_b), MMD_N, replace=False)].astype(np.float32)

def rbf_mmd(X, Y):
    """Unbiased MMD² estimate with RBF kernel (median-heuristic bandwidth)."""
    def sq_dists(A, B):
        return (np.sum(A**2, axis=1)[:, None]
                + np.sum(B**2, axis=1)[None, :]
                - 2 * (A @ B.T))

    dxy = sq_dists(X, Y)
    # Median heuristic on cross distances
    sigma2 = float(np.median(dxy[dxy > 0]))
    if sigma2 == 0:
        sigma2 = 1.0

    Kxx = np.exp(-sq_dists(X, X) / sigma2)
    Kyy = np.exp(-sq_dists(Y, Y) / sigma2)
    Kxy = np.exp(-dxy          / sigma2)

    n, m = len(X), len(Y)
    # Unbiased estimator: zero out diagonals
    np.fill_diagonal(Kxx, 0); np.fill_diagonal(Kyy, 0)
    mmd2 = (Kxx.sum() / (n*(n-1))
            + Kyy.sum() / (m*(m-1))
            - 2 * Kxy.mean())
    return float(np.sqrt(max(mmd2, 0))), float(np.sqrt(sigma2 / 2))

mmd_val, sigma = rbf_mmd(xa, xb)
print(f"MMD  = {mmd_val:.6f}")
print(f"σ    = {sigma:.4f}  (RBF bandwidth from median heuristic)")
print(f"\nInterpretation:")
print(f"  MMD ≈ 0    → distributions are identical")
print(f"  MMD > 0.1  → meaningful distributional shift")


## 9. UMAP scatter plot
Fits UMAP on 20 K samples per dataset (40 K total). PCA pre-reduction 768 → 50 dims speeds up nearest-neighbour search significantly.

In [ ]:
UMAP_N = 20_000

rng  = np.random.default_rng(7)
ua   = embs_a[rng.choice(len(embs_a), min(UMAP_N, len(embs_a)), replace=False)]
ub   = embs_b[rng.choice(len(embs_b), min(UMAP_N, len(embs_b)), replace=False)]
combined_umap = np.vstack([ua, ub])
labels_umap   = np.array(["A"] * len(ua) + ["B"] * len(ub))

print(f"PCA 768 → 50 …")
pca50  = PCA(n_components=50, random_state=42)
red50  = pca50.fit_transform(combined_umap)
print(f"Explained var (50 PCs): {pca50.explained_variance_ratio_.sum()*100:.1f} %")

print("UMAP 50 → 2 … (may take several minutes)")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                    metric="cosine", random_state=42, verbose=True)
coords = reducer.fit_transform(red50)

ua_2d = coords[:len(ua)]
ub_2d = coords[len(ua):]

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(ua_2d[:, 0], ua_2d[:, 1], s=2, alpha=0.2, color="#4C72B0",
           label="A (SinLlama)", rasterized=True)
ax.scatter(ub_2d[:, 0], ub_2d[:, 1], s=2, alpha=0.2, color="#DD8452",
           label="B (CPT-Dataset)", rasterized=True)
ax.set_title("UMAP Projection of HelaBERT CLS Embeddings (20 K per dataset, PCA 768→50→UMAP 2D)")
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.legend(markerscale=5, framealpha=0.9)
plt.tight_layout()
plt.savefig(OUT_DIR / "06_umap_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/06_umap_scatter.png")


## 10. UMAP density (hexbin)
Shows where probability mass is concentrated for each dataset. Side-by-side to reveal overlap and separation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
all_x = np.concatenate([ua_2d[:, 0], ub_2d[:, 0]])
all_y = np.concatenate([ua_2d[:, 1], ub_2d[:, 1]])
xlim  = (all_x.min() - 0.5, all_x.max() + 0.5)
ylim  = (all_y.min() - 0.5, all_y.max() + 0.5)

for ax, data, title, cmap in zip(
        axes,
        [ua_2d, ub_2d],
        ["A — SinLlama", "B — CPT-Dataset"],
        ["Blues", "Oranges"]):
    hb = ax.hexbin(data[:, 0], data[:, 1], gridsize=60, cmap=cmap,
                   mincnt=1, bins="log")
    plt.colorbar(hb, ax=ax, label="log(count)")
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_title(title)
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

fig.suptitle("UMAP Density — A vs B", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "07_umap_density.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/07_umap_density.png")


## 11. Cluster heatmap
Clusters the combined 50 K + 50 K reservoir embeddings with **KMeans** (cosine-style, on L2-normalised vectors), then shows the **A vs B composition of each cluster**. This is the most direct view of *whether the datasets occupy different semantic regions* — the spatial intuition behind the MMD number.

Both reservoir samples are equal-sized (50 K each), so a no-shift world would put **~50 % B in every cluster**. Clusters that deviate strongly reveal exactly where the corpora diverge.

In [ ]:
from sklearn.cluster import KMeans

N_CLUSTERS = 12

# Combine reservoir samples; cluster on L2-normalised vectors so Euclidean
# KMeans approximates cosine clustering (matches the rest of the notebook).
X  = np.vstack([embs_a, embs_b])
y  = np.array([0] * len(embs_a) + [1] * len(embs_b))   # 0 = A, 1 = B
Xn = normalize(X)

print(f"KMeans: {len(X):,} embeddings → {N_CLUSTERS} clusters …")
km       = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clusters = km.fit_predict(Xn)

# Per-cluster A / B composition
rows = []
for c in range(N_CLUSTERS):
    mask  = clusters == c
    n_a_c = int(((y == 0) & mask).sum())
    n_b_c = int(((y == 1) & mask).sum())
    total = n_a_c + n_b_c
    rows.append({"cluster": c, "size": total,
                 "pct_A": 100 * n_a_c / total,
                 "pct_B": 100 * n_b_c / total})

df = (pd.DataFrame(rows)
        .sort_values("pct_B", ascending=False)
        .reset_index(drop=True))

baseline_b = 100 * len(embs_b) / len(X)          # 50 % for equal samples
skew       = (df["pct_B"] - baseline_b).abs().mean()

# Heatmap: clusters × [A%, B%]
mat = df[["pct_A", "pct_B"]].values
fig, ax = plt.subplots(figsize=(6, 0.5 * N_CLUSTERS + 1.5))
sns.heatmap(mat, annot=True, fmt=".0f", cmap="RdBu_r", center=baseline_b,
            vmin=0, vmax=100, cbar_kws={"label": "% of cluster"},
            yticklabels=[f"C{int(r.cluster)}  (n={int(r.size):,})" for r in df.itertuples()],
            xticklabels=["A (SinLlama)", "B (CPT-Dataset)"], ax=ax)
ax.set_title(f"Cluster Composition — {N_CLUSTERS} KMeans clusters (50K A + 50K B)")
plt.tight_layout()
plt.savefig(OUT_DIR / "09_cluster_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nBaseline B-share (no shift) : {baseline_b:.1f} %")
print(f"Per-cluster B-share range   : {df.pct_B.min():.1f} % – {df.pct_B.max():.1f} %")
print(f"Mean |deviation| from 50 %  : {skew:.1f} pts")
print(f"\nMost B-skewed cluster: C{int(df.iloc[0].cluster)}  "
      f"({df.iloc[0].pct_B:.0f} % B, n={int(df.iloc[0].size):,})")
print(f"Most A-skewed cluster: C{int(df.iloc[-1].cluster)}  "
      f"({df.iloc[-1].pct_A:.0f} % A, n={int(df.iloc[-1].size):,})")
print("Saved → outputs/09_cluster_heatmap.png")

## 12. Pairwise cosine similarity distributions
`A→A` (within A), `B→B` (within B), `A→B` (cross). Tells us how similar embeddings are locally.

In [ ]:
N_PAIRS = 5_000
rng     = np.random.default_rng(99)

def sample_cosine_sims(X, Y=None, n=N_PAIRS):
    Xn = normalize(X)
    Yn = normalize(Y) if Y is not None else Xn
    i  = rng.integers(0, len(Xn), n)
    j  = rng.integers(0, len(Yn), n)
    return (Xn[i] * Yn[j]).sum(axis=1)

sim_aa = sample_cosine_sims(embs_a)
sim_bb = sample_cosine_sims(embs_b)
sim_ab = sample_cosine_sims(embs_a, embs_b)

print(f"Mean cosine sim  A→A: {sim_aa.mean():.4f}  ±{sim_aa.std():.4f}")
print(f"Mean cosine sim  B→B: {sim_bb.mean():.4f}  ±{sim_bb.std():.4f}")
print(f"Mean cosine sim  A→B: {sim_ab.mean():.4f}  ±{sim_ab.std():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(-0.1, 1.0, 80)
ax.hist(sim_aa, bins=bins, alpha=0.55, density=True, color="#4C72B0", label="A → A (within)")
ax.hist(sim_bb, bins=bins, alpha=0.55, density=True, color="#DD8452", label="B → B (within)")
ax.hist(sim_ab, bins=bins, alpha=0.55, density=True, color="#55A868", label="A → B (cross)")
ax.set_xlabel("Cosine similarity")
ax.set_ylabel("Density")
ax.set_title("Pairwise Cosine Similarity Distributions")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "08_similarity_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/08_similarity_histograms.png")


---

## 13. Notebook Summary

> **Roles:** **A = SinLlama-Dataset** — the corpus the SinLlama model was already trained on (the model's *existing* knowledge). **B = CPT-Dataset** — the *new* corpus for continual pre-training. The shift **A → B** is therefore *known → incoming*.

### What we did

| Step | Description |
|---|---|
| **1–2. Setup + model** | Loaded HelaBERT (110 M params) onto GPU, loaded SentencePiece tokenizer. |
| **3. Tokenisation** | SentencePiece → padded `input_ids` / `attention_mask` tensors with dynamic per-batch padding. |
| **4. Embedding pass** | Streamed both full corpora in batches of 512. Accumulated running centroid sum (exact, O(1) memory). Reservoir-sampled 50 K embeddings per dataset via Algorithm R. Saved checkpoints. Dataset A (SinLlama, larger corpus) was embedded on a cloud GPU and copied into `outputs/`; Dataset B (CPT) was embedded locally. |
| **5. Centroid distance** | Computed cosine similarity, cosine distance, L2 distance, and angle between the two full-corpus centroids. |
| **6. PCA centroid plot** | PCA projected 50 K points to 2D, overlaid centroid arrow A→B (`04_centroid_pca.png`). |
| **7. Compactness histogram** | Per-sample cosine distance from own centroid, showing semantic spread of each dataset (`05_centroid_distance_hist.png`). |
| **8. MMD** | Unbiased RBF-MMD² on 10 K subsamples, bandwidth from median heuristic. Quantifies full distributional shift. |
| **9. UMAP scatter** | PCA 768→50, then UMAP 50→2 on 20 K per dataset. Scatter coloured by dataset (`06_umap_scatter.png`). |
| **10. UMAP density** | Hexbin density maps side-by-side to show where each dataset concentrates (`07_umap_density.png`). |
| **11. Cluster heatmap** | KMeans (12 clusters) on 50 K + 50 K normalised embeddings; A/B composition per cluster (`09_cluster_heatmap.png`). Localises the MMD signal in semantic space. |
| **12. Pairwise similarity** | Sampled cosine similarities within A, within B, and cross A→B — local neighbourhood comparison (`08_similarity_histograms.png`). |

---

### Key findings

- **N_A** = 10,730,154 lines (SinLlama, known) &nbsp;•&nbsp; **N_B** = 8,696,658 lines (CPT-Dataset, new)
- Centroids computed over the **full** corpora; MMD / UMAP / clustering / pairwise sims over 50 K-per-dataset reservoir samples.

#### Centroid shift
| Metric | Value |
|---|---|
| Cosine similarity | **0.9944** |
| Cosine distance | 0.0056 |
| L2 distance | 2.5825 |
| Angle | 6.07° |

#### Distribution shift
| Metric | Value |
|---|---|
| MMD (RBF, median σ = 13.53) | **0.1176** |
| Cluster B-share range (12 clusters, 50 % = no shift) | **5 % – 83 %** |
| Mean cluster deviation from 50 % | **16.6 pts** |
| Most B-skewed cluster (CPT-heavy) | 83 % B (n≈9,000) |
| Most A-skewed cluster (SinLlama-heavy) | 95 % A (n≈4,200) |

#### Local similarity
| Pair | Mean cosine sim |
|---|---|
| A → A (SinLlama, within) | 0.7509 ± 0.071 |
| B → B (CPT, within) | 0.7680 ± 0.065 |
| A → B (cross) | 0.7529 ± 0.069 |

---

### What this means

- **The new data shares the model's semantic center.** Cosine similarity of 0.994 (centroids only 6° apart) means the CPT corpus is, on average, semantically aligned with what SinLlama already knows — consistent with NB1's near-perfect vocabulary overlap (Jaccard 0.9998). CPT will not drag the model into a foreign region.
- **Mild distributional shift, not a domain gap.** MMD = 0.118 sits right at the 0.1 "meaningful" threshold — detectable but small. Because the centroids coincide, MMD is capturing a difference in *shape/spread*, not location.
- **The new CPT data (B) is slightly more homogeneous than the SinLlama base (A).** Within-similarity is higher for B (0.768) than A (0.751), and B's embeddings sit closer to their own centroid (compactness 0.124 vs 0.134). The incoming corpus is a bit narrower than the broad base the model was trained on.
- **The shift is localised, not global** — the cluster heatmap is the key plot. The 12 clusters are **not** uniformly 50/50; they range from **5 % to 83 % B** (mean deviation 16.6 pts). Two pockets stand out:
  - **An 83 % CPT cluster (n≈9,000)** — content the new corpus emphasises that SinLlama barely covers. This is the **value of the CPT data**: genuinely new material the model will learn.
  - **A 95 % SinLlama cluster (n≈4,200)** — knowledge the model already has that the CPT data barely reinforces. This is the **catastrophic-forgetting-risk region**: during CPT the model sees little of it and may drift away from it.
- **No local domain gap.** Cross-dataset similarity A→B (0.753) sits *between* the two within-dataset values, so at the random-pair level new and known data are as similar to each other as within-corpus pairs. The structure only emerges once you cluster.

### Implication for continual pre-training

This is a **low-risk CPT scenario with one concrete thing to manage**:
- Shared vocabulary, shared semantic center, and overlapping local neighbourhoods → CPT on the new corpus will **not** catastrophically disrupt the model overall.
- The CPT data genuinely adds new content (the 83 % CPT cluster), so the continual-pretraining step is **worthwhile**, not redundant.
- **Recommendation: replay a light fraction of the SinLlama (A) data during CPT.** The 95 % SinLlama cluster is knowledge the model has but the CPT corpus under-reinforces — replaying old data specifically protects that region from forgetting. A heavy replay buffer or staged curriculum is *not* warranted: there is no broad gap to bridge, only one pocket to guard.

> **Caveat (not a problem):** Cell 2 reported `pooler.dense` weights newly initialised. This is harmless — embeddings are taken from `last_hidden_state[:, 0, :]` (raw CLS), never the pooler, so the random pooler weights are never used.
